In [19]:
import mappy
import edlib
import pysam
import os
import tempfile
import logging
from dataclasses import dataclass
from enum import Enum
from typing import Optional

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# --- 1. Configuration & Setup ---
good_bc = "ATGAGAATGCCGACC"

class MappingResult(Enum):
    SUCCESS = "success"
    NO_FLANKS_FOUND = "no_flanks_found"
    BARCODE_UNRECOGNISED = "barcode_unrecognised"
    BARCODE_AMBIGUOUS = "barcode_ambiguous"
    MAPPING_FAILED = "mapping_failed"


@dataclass
class PipelineStats:
    processed: int = 0
    no_flanks_found: int = 0
    barcode_unrecognised: int = 0
    barcode_ambiguous: int = 0
    mapping_failed: int = 0
    mapped: int = 0

    def log_summary(self):
        logger.info(
            f"Pipeline complete: {self.processed} reads processed | "
            f"{self.mapped} mapped | "
            f"{self.no_flanks_found} flanks missing | "
            f"{self.barcode_unrecognised} barcode unrecognised | "
            f"{self.barcode_ambiguous} barcode ambiguous | "
            f"{self.mapping_failed} mapping failed"
        )


def build_aligner_dict(plasmid_references: dict[str, dict[str, str]]) -> dict[str, mappy.Aligner]:
    """Initialises splice-aware mappy aligners using raw sequences in memory."""
    aligners = {}
    for barcode, ref_data in plasmid_references.items():
        # Pass the raw sequence string using seq=
        aligner = mappy.Aligner(seq=ref_data["seq"], preset="splice", k=14, w=5, min_chain_score=25)
        if not aligner:
            raise RuntimeError(f"Failed to load reference for barcode: {barcode}")
        
        # We can remove the warning about multiple seq_names here, 
        # because mappy doesn't assign names when loaded from raw strings.
        aligners[barcode] = aligner
    return aligners


# --- 2. Barcode Extraction & Matching ---

def extract_barcode_sequence(
    read_seq: str,
    f5_fwd: str,
    f3_fwd: str,
    f5_rev: str,
    f3_rev: str,
    search_window: int = 250,
    max_error_rate: float = 0.2,
    min_barcode_len: int = 4,
) -> Optional[str]:
    # ... inside the function, change the orientations list to use these arguments:

    """
    Searches the terminal ends of a read for flanking sequences and returns
    the raw extracted sequence between them (forward orientation).
    """
    seq_len = len(read_seq)
    search_window = min(search_window, seq_len)

    # Deduplicate windows if the read is shorter than the search window
    windows = [read_seq[:search_window]]
    if seq_len > search_window:
        windows.append(read_seq[-search_window:])

    err_5 = int(len(f5_fwd) * max_error_rate)
    err_3 = int(len(f3_fwd) * max_error_rate)

    orientations = [
        ("forward", f5_fwd, f3_fwd),
        ("reverse", f5_rev, f3_rev),
    ]

    for window_seq in windows:
        for orientation, f5_seq, f3_seq in orientations:
            res_5 = edlib.align(f5_seq, window_seq, mode="HW", task="locations", k=err_5)
            res_3 = edlib.align(f3_seq, window_seq, mode="HW", task="locations", k=err_3)

            if res_5["editDistance"] == -1 or res_3["editDistance"] == -1:
                continue

            end_5   = res_5["locations"][0][1]
            start_3 = res_3["locations"][-1][0]

            if start_3 <= end_5:
                continue

            extracted = window_seq[end_5 + 1 : start_3]

            if len(extracted) < min_barcode_len:
                continue

            if orientation == "reverse":
                extracted = mappy.revcomp(extracted)

            return extracted

    return None


def identify_library_barcode(
    extracted_seq: str,
    known_barcodes: list[str],
    max_edits: int = 2,
) -> tuple[Optional[str], MappingResult]:
    """
    Matches the extracted sequence against the known library barcodes using
    global (NW) alignment to tolerate Nanopore indel errors.
    """
    best_dist = max_edits
    best_matches: list[str] = []
    
    for bc in known_barcodes:
        # Pass current best_dist as k so edlib can short-circuit worse candidates
        res = edlib.align(extracted_seq, bc, mode="NW", k=best_dist)
        dist = res["editDistance"]

        if dist == -1:
            continue

        if dist < best_dist:
            best_dist = dist
            best_matches = [bc]
        elif dist == best_dist:
            best_matches.append(bc)

    if not best_matches:
        return None, MappingResult.BARCODE_UNRECOGNISED

    if len(best_matches) > 1:
        logger.debug(
            f"Ambiguous barcode: '{extracted_seq}' matches {best_matches} "
            f"at edit distance {best_dist}"
        )
        return None, MappingResult.BARCODE_AMBIGUOUS

    return best_matches[0], MappingResult.SUCCESS


# --- 3. BAM Handling ---

def create_bam_header(
    plasmid_references: dict[str, dict[str, str]]
) -> tuple[dict, dict[str, int]]:
    """Builds the unified SAM header explicitly using the rnames and sequence lengths."""
    header = {"HD": {"VN": "1.0", "SO": "unsorted"}, "SQ": []}
    ref_name_to_id: dict[str, int] = {}

    for i, (barcode, ref_data) in enumerate(sorted(plasmid_references.items())):
        ref_name = ref_data["rname"]
        ref_len  = len(ref_data["seq"])
        
        header["SQ"].append({"SN": ref_name, "LN": ref_len})
        ref_name_to_id[ref_name] = i

    return header, ref_name_to_id


def _build_aligned_segment(
    bam_writer: pysam.AlignmentFile,
    ref_id: int, 
    read_name: str,
    read_seq: str,
    aln: mappy.Alignment,
    cached_qual_array,
) -> pysam.AlignedSegment:
    """Constructs a pysam AlignedSegment, handling chimeras and hard-clips correctly."""
    a = pysam.AlignedSegment(bam_writer.header)
    a.query_name      = read_name
    a.reference_id    = ref_id  
    a.reference_start = aln.r_st
    a.mapping_quality = aln.mapq
    a.cigar = [(op, length) for length, op in aln.cigar]

    a.is_supplementary = not aln.is_primary
    a.is_reverse       = (aln.strand == -1)

    # Calculate the exact sequence length that this CIGAR string expects
    # pysam CIGAR ops that consume the query sequence: M(0), I(1), S(4), =(7), X(8)
    expected_seq_len = sum(length for op, length in a.cigar if op in (0, 1, 4, 7, 8))

    # If it's supplementary, OR if minimap2 hard-clipped the primary sequence,
    # drop the sequence string to prevent pysam from crashing.
    if a.is_supplementary or expected_seq_len != len(read_seq):
        a.query_sequence  = None
        a.query_qualities = None
    else:
        if a.is_reverse:
            a.query_sequence = mappy.revcomp(read_seq)
            if cached_qual_array is not None:
                a.query_qualities = cached_qual_array[::-1]
        else:
            a.query_sequence = read_seq
            if cached_qual_array is not None:
                a.query_qualities = cached_qual_array

    return a


def process_and_write_read(
    bam_writer: pysam.AlignmentFile,
    ref_id: int,  # <-- CHANGED THIS from ref_id_map
    read_name: str,
    read_seq: str,
    read_qual: Optional[str],
    barcode: str,
    aligners_dict: dict[str, mappy.Aligner],
) -> MappingResult:
    """Maps the read to its designated plasmid and writes all alignment hits."""
    aligner = aligners_dict.get(barcode)
    if not aligner:
        return MappingResult.BARCODE_UNRECOGNISED

    alignments = list(aligner.map(read_seq))
    if not alignments:
        return MappingResult.MAPPING_FAILED

    cached_qual_array = pysam.qualitystring_to_array(read_qual) if read_qual else None

    for aln in alignments:
        segment = _build_aligned_segment(
            bam_writer, ref_id, read_name, read_seq, aln, cached_qual_array # <-- Pass ref_id here
        )
        bam_writer.write(segment)

    return MappingResult.SUCCESS


# --- 4. Main Execution ---

def run_pipeline(
    fastq_path: str,
    plasmid_library: dict[str, dict[str, str]],
    output_bam_path: str,
    f5_fwd: str, # Pass the dynamically found flanks into the pipeline
    f3_fwd: str,
    barcode_max_edits: int = 2,
    log_interval: int = 10_000,
) -> PipelineStats:
    
    logger.info("Initialising aligners...")
    aligners = build_aligner_dict(plasmid_library)
    header, ref_map = create_bam_header(plasmid_library)
    known_barcodes = sorted(plasmid_library.keys())

    stats = PipelineStats()
    temp_bam_fd, temp_bam_path = tempfile.mkstemp(suffix=".bam")
    os.close(temp_bam_fd)

    f5_rev = mappy.revcomp(f3_fwd)
    f3_rev = mappy.revcomp(f5_fwd)

    x = 0

    try:
        logger.info("Processing reads...")
        with pysam.AlignmentFile(temp_bam_path, "wb", header=header) as unsorted_bam:
            for name, seq, qual in mappy.fastx_read(fastq_path):

                x += 1

                if x > 50:
                    continue

                stats.processed += 1
                if stats.processed % log_interval == 0:
                    logger.info(f"Processed {stats.processed} reads...")

                raw_extracted = extract_barcode_sequence(
                    seq, f5_fwd, f3_fwd, f5_rev, f3_rev
                )
                if not raw_extracted:
                    stats.no_flanks_found += 1
                    continue

                matched_barcode, match_result = identify_library_barcode(
                    raw_extracted, known_barcodes, max_edits=barcode_max_edits
                )
                
                if matched_barcode is None:
                    match match_result:
                        case MappingResult.BARCODE_UNRECOGNISED:
                            stats.barcode_unrecognised += 1
                        case MappingResult.BARCODE_AMBIGUOUS:
                            stats.barcode_ambiguous += 1
                    continue

                # ... inside the loop in run_pipeline
                correct_rname = plasmid_library[matched_barcode]["rname"]
                ref_id = ref_map[correct_rname]

                # --- FIX HERE ---
                # Pass 'ref_id' instead of 'ref_map'
                result = process_and_write_read(
                    unsorted_bam, ref_id, name, seq, qual, matched_barcode, aligners
                )
                # ----------------
                
                match result:
                    case MappingResult.SUCCESS:
                        stats.mapped += 1
                    case MappingResult.MAPPING_FAILED:
                        stats.mapping_failed += 1
                    case MappingResult.BARCODE_UNRECOGNISED:
                        stats.barcode_unrecognised += 1

        logger.info("Sorting and indexing final BAM file...")
        pysam.sort("-o", output_bam_path, temp_bam_path)
        pysam.index(output_bam_path)

    finally:
        if os.path.exists(temp_bam_path):
            os.remove(temp_bam_path)

    stats.log_summary()
    return stats

import pandas as pd
import mappy

def build_library_dictionary(csv_path: str, fasta_path: str, to_replace=None) -> dict[str, dict[str, str]]:
    """
    Combines the barcode-to-rname CSV with the multi-FASTA sequences.
    Handles cases where one rname is associated with multiple barcodes.
    Returns: { "BARCODE": {"rname": "Plasmid_1", "seq": "ATGC..."} }
    """
    # 1. Load CSV and map rname -> LIST of barcodes
    df = pd.read_csv(csv_path)
    rname_to_barcodes = {}
    for _, row in df.iterrows():
        rname_to_barcodes.setdefault(row.rname, []).append(row.barcode)
    
    plasmid_library = {}
    
    # 2. Parse the multi-FASTA
    for name, base_seq, _ in mappy.fastx_read(fasta_path):
        if name in rname_to_barcodes:
            # Iterate through EVERY barcode associated with this rname
            for barcode in rname_to_barcodes[name]:
                seq = base_seq
                
                if to_replace:
                    # .replace() returns a new string, so base_seq remains untouched for the next loop
                    seq = base_seq.replace(to_replace, barcode)
                
                plasmid_library[barcode] = {
                    "rname": name,
                    "seq": seq
                }
        else:
            print(f"Warning: '{name}' found in FASTA but has no barcode in the CSV.")
            
    return plasmid_library

def get_flanks(fasta_path: str, bc_placeholder: str, flank_len: int = 10) -> tuple[str, str]:
    """
    Scans a FASTA file to automatically determine the constant sequences 
    flanking the barcode placeholder.
    """
    lhs_flank_set = set()
    rhs_flank_set = set()
    
    for name, seq, _ in mappy.fastx_read(fasta_path):
        idx = seq.find(bc_placeholder)
        if idx == -1:
            raise ValueError(f"Placeholder '{bc_placeholder}' not found in sequence '{name}'.")
            
        lhs_flank = seq[idx - flank_len : idx]
        rhs_flank = seq[idx + len(bc_placeholder) : idx + len(bc_placeholder) + flank_len]
        
        lhs_flank_set.add(lhs_flank)
        rhs_flank_set.add(rhs_flank)
        
    if len(lhs_flank_set) != 1 or len(rhs_flank_set) != 1:
        raise ValueError(
            f"Inconsistent flanks detected across the FASTA!\n"
            f"LHS variants: {lhs_flank_set}\n"
            f"RHS variants: {rhs_flank_set}"
        )
        
    return lhs_flank_set.pop(), rhs_flank_set.pop()


# Usage:
# my_library = build_library_dictionary(
#     '/Users/ogw/.../v1_barcode_reference_linked.csv',
#     '/Users/ogw/.../all_plasmids.fasta'
# )

# Example usage:
# mock_library = {
#     "ACGTACGTACGT": "refs/plasmid_A.fasta",
#     "TGCATGCATGCA": "refs/plasmid_B.fasta",
# }
# stats = run_pipeline(
#     "data/nanopore_reads.fastq.gz",
#     mock_library,
#     "results/final_sorted.bam",
#     barcode_max_edits=2,
# )
fasta_file = '/Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2026/Matt_rotation/analysing_v1/V1_Opool_Reference_Library_Just_Inserts.fasta'
bc_placeholder = 'NNNNNNNNNNNNNNN'
csv_file = '/Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2026/Matt_rotation/analysing_v1/plasmids/v1_barcode_reference_linked.csv'

print("Detecting flanking sequences...")
f5, f3 = get_flanks(fasta_file, bc_placeholder, flank_len=10)
print(f"5' Flank: {f5}")
print(f"3' Flank: {f3}")

print("Building library dictionary...")
lib_dict = build_library_dictionary(csv_file, fasta_file, to_replace=bc_placeholder)

print(good_bc in lib_dict)

print("Starting pipeline...")
stats = run_pipeline(
    fastq_path="/Users/ogw/Downloads/C000356/1/demultiplexed/demultiplexed_DOX1.fastq.gz",
    plasmid_library=lib_dict,
    output_bam_path="/Users/ogw/Downloads/delete.bam",
    f5_fwd=f5,
    f3_fwd=f3,
    barcode_max_edits=2
)

2026-04-01 14:49:26,303 [INFO] Initialising aligners...


Detecting flanking sequences...
5' Flank: CGAgACGCGC
3' Flank: GTAAACTGGA
Building library dictionary...
True
Starting pipeline...


2026-04-01 14:49:27,175 [INFO] Processing reads...
2026-04-01 14:49:27,719 [INFO] Sorting and indexing final BAM file...
[E::sam_hrecs_refs_from_targets_array] Duplicate entry "Seq_18_125nt" in target list
[E::sam_hrecs_refs_from_targets_array] Duplicate entry "Seq_18_125nt" in target list


SamtoolsError: "samtools returned with error 1: stdout=, stderr=samtools sort: failed to change sort order header to 'SO:coordinate'\n\n"